# 29 — Quantum Fisher Scaling

**Repo:** `decoherence-free-sensing`  
**Notebook role:** precision-scaling layer  
**Previous:** `23_lieb_mattis_states.ipynb`  
**Next:** `37_two_body_measurement.ipynb`

Notebook 23 specified admissible DFS states:

\[
J_z^+|\psi\rangle = 0,
\qquad
\mathrm{Var}_\psi(J_z^-)>0.
\]

Notebook 29 quantifies what that retained differential sensitivity provides.

For a pure state and differential phase generator \(J_z^-\),

\[
F_Q[|\psi\rangle,J_z^-]=4\,\mathrm{Var}_\psi(J_z^-).
\]

The quantum Cramér-Rao bound gives

\[
\Delta^2\varphi \geq \frac{1}{F_Q}.
\]

For the Lieb-Mattis scaling reference, we use

\[
\Delta^2\varphi_{\rm LM}
\sim
\frac{3}{N^2+4N}.
\]

## Engineering statement

Quantum Fisher information quantifies how much differential sensitivity remains after the DFS constraint has specified admissible states.


## Notebook outputs

Running this notebook creates:

- `results/figures/29_qfi_pipeline.png`
- `results/figures/29_generator_variance_to_qfi.png`
- `results/figures/29_sql_hl_lieb_mattis_scaling_v2.png`
- `results/figures/29_precision_gain_over_sql.png`
- `results/figures/29_qfi_design_rule.png`
- `results/qfi_scaling_summary.csv`
- `results/qfi_scaling_summary.json`
- `results/decoherence_free_sensing_qfi_scaling_outputs.zip`


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
SITE = ROOT / "site"

for path in [RESULTS, FIGURES, SITE]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
})


## 1. QFI pipeline

Notebook 23 established the state-design condition:

\[
J_z^+|\psi\rangle=0,
\qquad
\mathrm{Var}(J_z^-)>0.
\]

Notebook 29 converts that second condition into a metrological quantity.

\[
\mathrm{Var}(J_z^-)
\longrightarrow
F_Q
\longrightarrow
\Delta^2\varphi.
\]


In [ ]:
def draw_box(ax, xy, width, height, title, body=None, fontsize=15, facecolor="white", edgecolor="black", lw=1.8):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.025,rounding_size=0.04",
        linewidth=lw,
        edgecolor=edgecolor,
        facecolor=facecolor
    )
    ax.add_patch(box)
    ax.text(
        x + width/2,
        y + height*0.62 if body else y + height/2,
        title,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold"
    )
    if body:
        ax.text(
            x + width/2,
            y + height*0.28,
            body,
            ha="center",
            va="center",
            fontsize=11
        )

def draw_arrow(ax, p0, p1):
    ax.add_patch(FancyArrowPatch(
        p0, p1,
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=1.8
    ))

fig, ax = plt.subplots(figsize=(13.0, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "QFI Pipeline", ha="center", va="center", fontsize=23)

items = [
    (0.05, "DFS state", r"$J_z^+=0$" + "\n" + r"$\mathrm{Var}(J_z^-)>0$"),
    (0.285, "Variance", r"differential" + "\n" + r"generator spread"),
    (0.52, "QFI", r"$F_Q=4\,\mathrm{Var}(J_z^-)$"),
    (0.755, "Precision", r"$\Delta^2\varphi\geqq 1/F_Q$"),
]
w, h, y = 0.19, 0.32, 0.42

for x0, title, body in items:
    draw_box(ax, (x0, y), w, h, title, body)

for x0 in [0.245, 0.48, 0.715]:
    draw_arrow(ax, (x0, y + h/2), (x0 + 0.035, y + h/2))

ax.text(
    0.5,
    0.22,
    "QFI quantifies the differential sensitivity retained inside the DFS.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "29_qfi_pipeline.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 2. From generator variance to QFI

For a pure state under a unitary phase shift,

\[
|\psi_\varphi\rangle
=
e^{-i\varphi G}|\psi\rangle,
\]

the quantum Fisher information is

\[
F_Q = 4\,\mathrm{Var}_\psi(G).
\]

For differential sensing,

\[
G=J_z^-.
\]

Thus, after the DFS constraint removes the common coordinate, the remaining resource is the variance of the differential generator.


In [ ]:
var_values = np.linspace(0, 25, 100)
qfi_values = 4 * var_values
crb_values = np.where(qfi_values > 0, 1 / qfi_values, np.nan)

fig, ax = plt.subplots(figsize=(8.4, 5.8))
ax.plot(var_values, qfi_values, linewidth=2.2, label=r"$F_Q=4\,\mathrm{Var}(J_z^-)$")
ax.set_title("Generator Variance Specifies Quantum Fisher Information")
ax.set_xlabel(r"differential generator variance $\mathrm{Var}(J_z^-)$")
ax.set_ylabel(r"quantum Fisher information $F_Q$")
ax.grid(True, alpha=0.3)
ax.legend()

ax.text(
    0.55,
    0.22,
    r"more retained spread in $J_z^-$" + "\n" + r"$\Rightarrow$ larger QFI",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=13,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = FIGURES / "29_generator_variance_to_qfi.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 3. Scaling references

We compare three variance references for total atom number \(N\):

Standard quantum limit:

\[
\Delta^2\varphi_{\rm SQL}\sim \frac{1}{N}.
\]

Heisenberg reference:

\[
\Delta^2\varphi_{\rm HL}\sim \frac{1}{N^2}.
\]

Lieb-Mattis DFS-compatible reference:

\[
\Delta^2\varphi_{\rm LM}
\sim
\frac{3}{N^2+4N}.
\]

The Lieb-Mattis curve approaches Heisenberg-like scaling up to a constant factor while respecting the DFS constraint.


In [ ]:
N = np.unique(np.round(np.logspace(np.log10(4), np.log10(2000), 220)).astype(int))

sql = 1 / N
hl = 1 / (N**2)
lm = 3 / (N**2 + 4*N)

fig, ax = plt.subplots(figsize=(8.6, 6.0))

ax.loglog(N, sql, linewidth=2.2, label="SQL: 1/N")
ax.loglog(N, hl, linewidth=2.2, label="Heisenberg: 1/N²")
ax.loglog(N, lm, linewidth=2.2, label="Lieb-Mattis QCRB: 3/(N²+4N)")

ax.set_title("Scaling References for Differential Quantum Sensing")
ax.set_xlabel("total atom number N")
ax.set_ylabel(r"variance reference $\Delta^2\varphi$")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

path = FIGURES / "29_sql_hl_lieb_mattis_scaling_v2.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 4. Precision gain over SQL

The SQL reference is a baseline:

\[
\Delta^2\varphi_{\rm SQL}=\frac{1}{N}.
\]

The Lieb-Mattis gain over SQL is

\[
\frac{\Delta^2\varphi_{\rm SQL}}
{\Delta^2\varphi_{\rm LM}}
=
\frac{N^2+4N}{3N}
=
\frac{N+4}{3}.
\]

So the DFS-compatible Lieb-Mattis reference improves linearly over SQL while retaining common-mode rejection.


In [ ]:
gain_lm_over_sql = sql / lm
gain_hl_over_sql = sql / hl

fig, ax = plt.subplots(figsize=(8.4, 5.8))

ax.loglog(N, gain_lm_over_sql, linewidth=2.2, label="Lieb-Mattis gain over SQL")
ax.loglog(N, gain_hl_over_sql, linestyle="--", linewidth=2.2, label="Heisenberg gain over SQL")

ax.set_title("Precision Gain Over SQL")
ax.set_xlabel("total atom number N")
ax.set_ylabel(r"gain: $\Delta^2\varphi_{\rm SQL}/\Delta^2\varphi$")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

ax.text(
    0.56,
    0.22,
    r"Lieb-Mattis gain $\sim (N+4)/3$",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=13,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = FIGURES / "29_precision_gain_over_sql.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 5. QFI design rule

The design rule is:

1. satisfy the DFS constraint,
2. retain differential generator variance,
3. compute QFI,
4. translate QFI into a precision bound,
5. compare against SQL and Heisenberg references.

Notebook 23 specified useful DFS states. Notebook 29 quantifies them.


In [ ]:
fig, ax = plt.subplots(figsize=(13.0, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "QFI Design Rule", ha="center", va="center", fontsize=23)

items = [
    (0.05, "Constrain", r"fix $J_z^+$" + "\nDFS admissible"),
    (0.285, "Retain", r"spread in" + "\n" + r"$J_z^-$"),
    (0.52, "Quantify", r"$F_Q=4\,\mathrm{Var}$"),
    (0.755, "Compare", r"SQL / HL /" + "\n" + r"Lieb-Mattis"),
]
w, h, y = 0.19, 0.32, 0.42

for x0, title, body in items:
    draw_box(ax, (x0, y), w, h, title, body)

for x0 in [0.245, 0.48, 0.715]:
    draw_arrow(ax, (x0, y + h/2), (x0 + 0.035, y + h/2))

ax.text(
    0.5,
    0.22,
    "A useful DFS state becomes a precision statement through quantum Fisher information.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "29_qfi_design_rule.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 6. Summary table


In [ ]:
summary = pd.DataFrame([
    {
        "item": "pure_state_qfi",
        "value": "F_Q = 4 Var(G)",
        "engineering_role": "turns generator spread into sensitivity"
    },
    {
        "item": "differential_generator",
        "value": "G = J_z^-",
        "engineering_role": "parameter generator after DFS constraint removes common coordinate"
    },
    {
        "item": "cramer_rao_bound",
        "value": "Delta^2 varphi >= 1/F_Q",
        "engineering_role": "converts sensitivity into a precision lower bound"
    },
    {
        "item": "sql_reference",
        "value": "Delta^2 varphi_SQL = 1/N",
        "engineering_role": "classical independent-particle baseline"
    },
    {
        "item": "heisenberg_reference",
        "value": "Delta^2 varphi_HL = 1/N^2",
        "engineering_role": "ideal quadratic scaling reference"
    },
    {
        "item": "lieb_mattis_reference",
        "value": "Delta^2 varphi_LM = 3/(N^2+4N)",
        "engineering_role": "DFS-compatible entangled scaling reference"
    },
    {
        "item": "gain_over_sql",
        "value": "SQL/LM = (N+4)/3",
        "engineering_role": "linear gain while respecting DFS constraint"
    },
    {
        "item": "design_rule",
        "value": "Constrain first; quantify retained differential sensitivity second",
        "engineering_role": "QFI specification"
    },
])

summary_csv = RESULTS / "qfi_scaling_summary.csv"
summary_json = RESULTS / "qfi_scaling_summary.json"

summary.to_csv(summary_csv, index=False)
summary.to_json(summary_json, orient="records", indent=2)

summary


## 7. Export notebook outputs

This cell creates one zip archive containing all generated figures and summary files.


In [ ]:
zip_path = RESULTS / "decoherence_free_sensing_qfi_scaling_outputs.zip"

public_figures = [
    FIGURES / "29_qfi_pipeline.png",
    FIGURES / "29_generator_variance_to_qfi.png",
    FIGURES / "29_sql_hl_lieb_mattis_scaling_v2.png",
    FIGURES / "29_precision_gain_over_sql.png",
    FIGURES / "29_qfi_design_rule.png",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in public_figures:
        if file.exists():
            zf.write(file, file.relative_to(ROOT))
    zf.write(summary_csv, summary_csv.relative_to(ROOT))
    zf.write(summary_json, summary_json.relative_to(ROOT))

print(f"Created: {zip_path}")

# Optional Colab download:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

zip_path


## 8. Next notebook

Suggested next notebook:

`37_two_body_measurement.ipynb`

Core goal:

- connect the QFI scaling reference to an observable measurement,
- identify the two-body fringe response,
- show why the operating point \(\varphi=\pi/4\) maximizes slope,
- translate state sensitivity into a measurement protocol.
